In [9]:
%load_ext autoreload
%autoreload 2

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import h5py
import os
from multiprocess import Pool
import importlib
from tqdm.auto import tqdm
import warnings
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)

workspace_root = os.getcwd()  
sys.path.insert(0, workspace_root + "/../../../../")
import pyanalib.pandas_helpers as ph
from makedf.util import *

workspace_root = os.getcwd()  
sys.path.insert(0, workspace_root + "/../../")
import kinematics
import gump_cuts as gc
import loaddf
import syst

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
RECO = "PANDORA"
FONTSIZE = 14
plt.style.use('/exp/sbnd/app/users/nrowe/cafpyana/analysis_village/gump/dune.mplstyle')

In [36]:
DF_DIR = "/exp/sbnd/data/users/gputnam/GUMP/sbn-rewgted-9/"

SCV_FILES = [DF_DIR + "SBNDMCCV_%i.df" % i for i in range(2)]
SDIRT_FILE = DF_DIR + "SBND_SpringLowEMC.df"
SBEAMOFF_FILE = DF_DIR + "SBND_SpringBNBOffData.df"

SDETVAR_FILES = [
    [DF_DIR + "SBNDMCCV_%i.df" % i for i in range(2)],
    [DF_DIR + "SBND_SpringMC_WMXThetaXW.df"],
    [DF_DIR + "SBND_SpringMC_WMYZ.df"],
]
# SDETVAR_FILES = [
#     [DF_DIR + "SBND_SpringMC_Nom.df"],
#     [DF_DIR + "SBND_SpringMC_WMXThetaXW.df"],
#     [DF_DIR + "SBND_SpringMC_WMYZ.df"],
# ]

SDETVAR_NAMES = ["Nominal",
                 "WM $X\\theta_{xw}$", "WM $YZ$", 
                 ]

SDETVAR_FILES_SMALL = [
    DF_DIR + "SBND_SpringMC_Nom.df",
    DF_DIR + "SBND_SpringMC_2xSCE.df",
]

SDETVAR_NAMES_SMALL = ["Nominal", "2xSCE"]

IRUN2_CV_FILES = [DF_DIR + "ICARUSRun2_SpringMCOverlay_rewgt.df"]
IRUN4_CV_FILES = [DF_DIR + "ICARUSRun4_SpringMCOverlay_rewgt_%i.df" % i for i in range(2)]
IRUN2_DIRT_FILES = [DF_DIR + "ICARUS_Spring_Overlay_Dirt.df"]
IRUN4_DIRT_FILES = [DF_DIR + "ICARUSRun4_Spring_Overlay_Dirt.df"]
IRUN2_BEAMOFF_FILES = [DF_DIR + "ICARUS_SpringRun2BNBOff_unblind.df"]
IRUN4_BEAMOFF_FILES = [DF_DIR + "ICARUS_SpringRun4BNBOff_unblind.df"]
IRUN2_DETVAR_FILES = [[DF_DIR + "ICARUSRun2_SpringMCOverlay_rewgt.df"],
                      [DF_DIR + "ICARUSRun2_Spring_Overlay_WMXThXW.df"],
                      [DF_DIR + "ICARUSRun2_Spring_Overlay_SCE.df"]]
print("REMINDER, YOU ARE MISSING ICARUS RUN 4 WIREMOD STILL!!!")
IRUN4_DETVAR_FILES = [[DF_DIR + "ICARUSRun4_SpringMCOverlay_rewgt_%i.df" % i for i in range(2)],
                      [DF_DIR + "ICARUSRun4_SpringMCOverlay_rewgt_%i.df" % i for i in range(2)],
                      [DF_DIR + "ICARUSRun4_Spring_Overlay_SCE.df"]]

IDETVAR_NAMES = ["Nominal", 
                 "WM $X\\theta_{xw}$",
                 "SCE"]

IRUN2_POT = 2e20
IRUN4_POT = 3e20
IGOAL_POT = IRUN2_POT + IRUN4_POT
SGOAL_POT = 1e20

REMINDER, YOU ARE MISSING ICARUS RUN 4 WIREMOD STILL!!!


In [19]:
%%capture
# Load Run 2 and Run 4 ICARUS MC separately
Idf_r2, Imatch_r2, Ipot_r2 = loaddf.loadl(IRUN2_CV_FILES, njob=min(len(IRUN2_CV_FILES), 20), detector="ICARUS Run2", 
                                 preselection=gc.presel_cut, reweight_aFF=True, drops=loaddf.get_std_drops(), lightmem=True)
# Scale each run to its target POT before combining
loaddf.scale_pot(Idf_r2, Ipot_r2, IRUN2_POT)

# Load Run 2 and Run 4 ICARUS dirt separately
if len(IRUN2_DIRT_FILES) > 0:
    Idirt_r2, Idirtmatch_r2, Idirtpot_r2 = loaddf.loadl(IRUN2_DIRT_FILES, njob=min(len(IRUN2_DIRT_FILES), 20), detector="ICARUS Run2", 
                                               preselection=gc.presel_cut, include_syst=False, drops=loaddf.get_std_drops(), lightmem=True)
    loaddf.scale_pot(Idirt_r2, Idirtpot_r2, IRUN2_POT)

# Load Run 2 and Run 4 ICARUS beamoff separately
Ioffbeam_r2, _, Ioffbeampot_r2 = loaddf.loadl(IRUN2_BEAMOFF_FILES, detector="ICARUS Run2", offbeampot=True, 
                                                   preselection=gc.presel_cut, include_syst=False, load_truth=False, match_Enu = False, drops=loaddf.get_std_drops(), lightmem=True)
loaddf.scale_pot(Ioffbeam_r2, Ioffbeampot_r2, IRUN2_POT)

In [28]:
Idetvars_r2, Idetvarsmatch_r2, Idetvar_pots_r2 = zip(*tqdm([loaddf.loadl(f, preselection=gc.presel_cut, include_syst=False, detector="ICARUS Run2", njob=min(len(f), 20)) for f in IRUN2_DETVAR_FILES]))

  0%|          | 0/1 [00:00<?, ?it/s]

[ICARUSRun2_SpringMCOverlay_rewgt.df idf=0] dedup: dropped 0 duplicated ('run', 'evt', 'nu_E0') keys (0 hdr rows)
[ICARUSRun2_SpringMCOverlay_rewgt.df idf=1] dedup: dropped 0 duplicated ('run', 'evt', 'nu_E0') keys (0 hdr rows)
[ICARUSRun2_SpringMCOverlay_rewgt.df idf=2] dedup: dropped 0 duplicated ('run', 'evt', 'nu_E0') keys (0 hdr rows)
[ICARUSRun2_SpringMCOverlay_rewgt.df idf=3] dedup: dropped 0 duplicated ('run', 'evt', 'nu_E0') keys (0 hdr rows)
[ICARUSRun2_SpringMCOverlay_rewgt.df idf=4] dedup: dropped 0 duplicated ('run', 'evt', 'nu_E0') keys (0 hdr rows)
[ICARUSRun2_SpringMCOverlay_rewgt.df idf=5] dedup: dropped 0 duplicated ('run', 'evt', 'nu_E0') keys (0 hdr rows)
[ICARUSRun2_SpringMCOverlay_rewgt.df idf=6] dedup: dropped 0 duplicated ('run', 'evt', 'nu_E0') keys (0 hdr rows)
[ICARUSRun2_SpringMCOverlay_rewgt.df idf=7] dedup: dropped 0 duplicated ('run', 'evt', 'nu_E0') keys (0 hdr rows)
[ICARUSRun2_SpringMCOverlay_rewgt.df idf=8] dedup: dropped 0 duplicated ('run', 'evt', '

  0%|          | 0/1 [00:00<?, ?it/s]

[ICARUSRun2_Spring_Overlay_WMXThXW.df idf=0] dedup: dropped 0 duplicated ('run', 'evt', 'nu_E0') keys (0 hdr rows)
dedup: dropped 750260 duplicated rows 1.0 of the POT remaining. Before POT: 1.0313544324903516e+21, after POT 1.0313544324903516e+21.


  0%|          | 0/1 [00:00<?, ?it/s]

[ICARUSRun2_Spring_Overlay_SCE.df idf=0] dedup: dropped 0 duplicated ('run', 'evt', 'nu_E0') keys (0 hdr rows)
dedup: dropped 844728 duplicated rows 1.0 of the POT remaining. Before POT: 1.1692749896475036e+21, after POT 1.1692749896475036e+21.


  0%|          | 0/3 [00:00<?, ?it/s]

In [29]:
%%capture
Idf_r4, Imatch_r4, Ipot_r4 = loaddf.loadl(IRUN4_CV_FILES, njob=min(len(IRUN4_CV_FILES), 20), detector="ICARUS Run4", 
                                 preselection=gc.presel_cut, reweight_aFF=True, drops=loaddf.get_std_drops(), lightmem=True)

# Scale each run to its target POT before combining
loaddf.scale_pot(Idf_r4, Ipot_r4, IRUN4_POT)

# Load Run 2 and Run 4 ICARUS dirt separately
if len(IRUN4_DIRT_FILES) > 0:
    Idirt_r4, Idirtmatch_r4, Idirtpot_r4 = loaddf.loadl(IRUN4_DIRT_FILES, njob=min(len(IRUN4_DIRT_FILES), 20), detector="ICARUS Run4", 
                                               preselection=gc.presel_cut, include_syst=False, drops=loaddf.get_std_drops(), lightmem=True)
    loaddf.scale_pot(Idirt_r4, Idirtpot_r4, IRUN4_POT)

Ioffbeam_r4, _, Ioffbeampot_r4 = loaddf.loadl(IRUN4_BEAMOFF_FILES, detector="ICARUS Run4", offbeampot=True, 
                                                   preselection=gc.presel_cut, include_syst=False, load_truth=False, match_Enu = False, drops=loaddf.get_std_drops(), lightmem=True)
loaddf.scale_pot(Ioffbeam_r4, Ioffbeampot_r4, IRUN4_POT)

In [30]:
Idetvars_r4, Idetvarsmatch_r4, Idetvar_pots_r4 = zip(*tqdm([loaddf.loadl(f, njob=min(len(f), 20), preselection=gc.presel_cut, include_syst=False, detector="ICARUS Run4") for f in IRUN4_DETVAR_FILES]))

  0%|          | 0/2 [00:00<?, ?it/s]

[ICARUSRun4_SpringMCOverlay_rewgt_1.df idf=0] dedup: dropped 0 duplicated ('run', 'evt', 'nu_E0') keys (0 hdr rows)
[ICARUSRun4_SpringMCOverlay_rewgt_0.df idf=0] dedup: dropped 0 duplicated ('run', 'evt', 'nu_E0') keys (0 hdr rows)
[ICARUSRun4_SpringMCOverlay_rewgt_1.df idf=1] dedup: dropped 0 duplicated ('run', 'evt', 'nu_E0') keys (0 hdr rows)
[ICARUSRun4_SpringMCOverlay_rewgt_0.df idf=1] dedup: dropped 0 duplicated ('run', 'evt', 'nu_E0') keys (0 hdr rows)
[ICARUSRun4_SpringMCOverlay_rewgt_1.df idf=2] dedup: dropped 0 duplicated ('run', 'evt', 'nu_E0') keys (0 hdr rows)
[ICARUSRun4_SpringMCOverlay_rewgt_0.df idf=2] dedup: dropped 0 duplicated ('run', 'evt', 'nu_E0') keys (0 hdr rows)
[ICARUSRun4_SpringMCOverlay_rewgt_0.df idf=3] dedup: dropped 0 duplicated ('run', 'evt', 'nu_E0') keys (0 hdr rows)
[ICARUSRun4_SpringMCOverlay_rewgt_1.df idf=3] dedup: dropped 0 duplicated ('run', 'evt', 'nu_E0') keys (0 hdr rows)
[ICARUSRun4_SpringMCOverlay_rewgt_0.df idf=4] dedup: dropped 0 duplicate

  0%|          | 0/2 [00:00<?, ?it/s]

[ICARUSRun4_SpringMCOverlay_rewgt_0.df idf=0] dedup: dropped 0 duplicated ('run', 'evt', 'nu_E0') keys (0 hdr rows)[ICARUSRun4_SpringMCOverlay_rewgt_1.df idf=0] dedup: dropped 0 duplicated ('run', 'evt', 'nu_E0') keys (0 hdr rows)

[ICARUSRun4_SpringMCOverlay_rewgt_0.df idf=1] dedup: dropped 0 duplicated ('run', 'evt', 'nu_E0') keys (0 hdr rows)
[ICARUSRun4_SpringMCOverlay_rewgt_1.df idf=1] dedup: dropped 0 duplicated ('run', 'evt', 'nu_E0') keys (0 hdr rows)
[ICARUSRun4_SpringMCOverlay_rewgt_0.df idf=2] dedup: dropped 0 duplicated ('run', 'evt', 'nu_E0') keys (0 hdr rows)
[ICARUSRun4_SpringMCOverlay_rewgt_1.df idf=2] dedup: dropped 0 duplicated ('run', 'evt', 'nu_E0') keys (0 hdr rows)
[ICARUSRun4_SpringMCOverlay_rewgt_0.df idf=3] dedup: dropped 0 duplicated ('run', 'evt', 'nu_E0') keys (0 hdr rows)
[ICARUSRun4_SpringMCOverlay_rewgt_1.df idf=3] dedup: dropped 0 duplicated ('run', 'evt', 'nu_E0') keys (0 hdr rows)
[ICARUSRun4_SpringMCOverlay_rewgt_0.df idf=4] dedup: dropped 0 duplicate

  0%|          | 0/1 [00:00<?, ?it/s]

[ICARUSRun4_Spring_Overlay_SCE.df idf=0] dedup: dropped 0 duplicated ('run', 'evt', 'nu_E0') keys (0 hdr rows)
[ICARUSRun4_Spring_Overlay_SCE.df idf=1] dedup: dropped 0 duplicated ('run', 'evt', 'nu_E0') keys (0 hdr rows)
dedup: dropped 2225242 duplicated rows 1.0 of the POT remaining. Before POT: 2.2022693657209156e+21, after POT 2.2022693657209156e+21.


  0%|          | 0/3 [00:00<?, ?it/s]

In [34]:
%%capture
Sdf, Smatch, Spot = loaddf.loadl(SCV_FILES, njob=min(len(SCV_FILES), 20), detector="SBND",
                                 preselection=gc.presel_cut, reweight_aFF=True, drops=loaddf.get_std_drops(), lightmem=True)
Sdirt, Sdirtmatch, Sdirtpot = loaddf.load(SDIRT_FILE, detector="SBND",
                                          preselection=gc.presel_cut, include_syst=False, drops=loaddf.get_std_drops(), lightmem=True)
Soffbeam, Soffbeammatch, Soffbeampot = loaddf.load(SBEAMOFF_FILE, detector="SBND",
                                                   preselection=gc.presel_cut, offbeampot=True, include_syst=False, load_truth=False, drops=loaddf.get_std_drops(), lightmem=True)

In [37]:
Sdetvars_small, Sdetvarsmatch_small, Sdetvar_pots_small = zip(*tqdm([loaddf.load(f, preselection=gc.presel_cut, include_syst=False, detector="SBND") for f in SDETVAR_FILES_SMALL]))
Sdetvars, Sdetvarsmatch, Sdetvar_pots = zip(*tqdm([loaddf.loadl(f, preselection=gc.presel_cut, include_syst=False, detector="SBND", njob=min(len(f), 20)) for f in SDETVAR_FILES]))

[SBND_SpringMC_Nom.df idf=0] dedup: dropped 5 duplicated ('run', 'evt', 'nu_E0') keys (10 hdr rows)
[SBND_SpringMC_2xSCE.df idf=0] dedup: dropped 5 duplicated ('run', 'evt', 'nu_E0') keys (10 hdr rows)


  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

[SBNDMCCV_1.df idf=0] dedup: dropped 79 duplicated ('run', 'evt', 'nu_E0') keys (158 hdr rows)
[SBNDMCCV_0.df idf=0] dedup: dropped 0 duplicated ('run', 'evt', 'nu_E0') keys (0 hdr rows)
[SBNDMCCV_1.df idf=1] dedup: dropped 15 duplicated ('run', 'evt', 'nu_E0') keys (30 hdr rows)
[SBNDMCCV_0.df idf=1] dedup: dropped 0 duplicated ('run', 'evt', 'nu_E0') keys (0 hdr rows)
[SBNDMCCV_1.df idf=2] dedup: dropped 20 duplicated ('run', 'evt', 'nu_E0') keys (40 hdr rows)
[SBNDMCCV_0.df idf=2] dedup: dropped 0 duplicated ('run', 'evt', 'nu_E0') keys (0 hdr rows)
[SBNDMCCV_1.df idf=3] dedup: dropped 42 duplicated ('run', 'evt', 'nu_E0') keys (84 hdr rows)
[SBNDMCCV_0.df idf=3] dedup: dropped 0 duplicated ('run', 'evt', 'nu_E0') keys (0 hdr rows)
[SBNDMCCV_1.df idf=4] dedup: dropped 16 duplicated ('run', 'evt', 'nu_E0') keys (32 hdr rows)
[SBNDMCCV_0.df idf=4] dedup: dropped 0 duplicated ('run', 'evt', 'nu_E0') keys (0 hdr rows)
[SBNDMCCV_1.df idf=5] dedup: dropped 12 duplicated ('run', 'evt', 'nu

  0%|          | 0/1 [00:00<?, ?it/s]

[SBND_SpringMC_WMXThetaXW.df idf=0] dedup: dropped 16 duplicated ('run', 'evt', 'nu_E0') keys (32 hdr rows)
[SBND_SpringMC_WMXThetaXW.df idf=1] dedup: dropped 0 duplicated ('run', 'evt', 'nu_E0') keys (0 hdr rows)
[SBND_SpringMC_WMXThetaXW.df] cross-idf dedup: dropped 2 duplicated ('run', 'evt', 'nu_E0') keys (4 match rows)
dedup: dropped 1582839 duplicated rows 1.0 of the POT remaining. Before POT: 6.037840208739361e+20, after POT 6.037840208739361e+20.


  0%|          | 0/1 [00:00<?, ?it/s]

[SBND_SpringMC_WMYZ.df idf=0] dedup: dropped 32 duplicated ('run', 'evt', 'nu_E0') keys (64 hdr rows)
[SBND_SpringMC_WMYZ.df idf=1] dedup: dropped 1 duplicated ('run', 'evt', 'nu_E0') keys (2 hdr rows)
[SBND_SpringMC_WMYZ.df] cross-idf dedup: dropped 3 duplicated ('run', 'evt', 'nu_E0') keys (6 match rows)
dedup: dropped 1582482 duplicated rows 1.0 of the POT remaining. Before POT: 6.039144141568973e+20, after POT 6.039144141568973e+20.


  0%|          | 0/3 [00:00<?, ?it/s]

In [38]:
Idf = pd.concat([Idf_r2, Idf_r4]).reset_index(drop=True)
Imatch = pd.concat([Imatch_r2, Imatch_r4])
Ipot = IRUN2_POT + IRUN4_POT

# Load Run 2 and Run 4 ICARUS dirt separately
if len(IRUN2_DIRT_FILES) > 0 and len(IRUN4_DIRT_FILES) > 0:
    Idirt = pd.concat([Idirt_r2, Idirt_r4]).reset_index(drop=True)
else:
    Idirt = pd.DataFrame(columns=Idf.columns)
Idirtpot = IRUN2_POT + IRUN4_POT
Ioffbeam = pd.concat([Ioffbeam_r2, Ioffbeam_r4]).reset_index(drop=True)
Ioffbeampot = IRUN2_POT + IRUN4_POT

Idetvars_r2, Idetvar_pots_r2 = loaddf.match_common_evts(Idetvarsmatch_r2, Idetvars_r2, Idetvar_pots_r2)
Idetvars_r4, Idetvar_pots_r4 = loaddf.match_common_evts(Idetvarsmatch_r4, Idetvars_r4, Idetvar_pots_r4)

for i in range(len(Idetvars_r2)):
    loaddf.scale_pot(Idetvars_r2[i], Idetvar_pots_r2[i], IRUN2_POT)

for i in range(len(Idetvars_r4)):
    loaddf.scale_pot(Idetvars_r4[i], Idetvar_pots_r4[i], IRUN4_POT)

Idetvars = [pd.concat([r2, r4]) for r2, r4 in zip(Idetvars_r2, Idetvars_r4)]

(np.float32(1.01182035e+21), np.float32(0.19766355))

(np.float32(1.0118225e+21), np.float32(0.19766313))

(np.float32(1.0118585e+21), np.float32(0.1976561))

(np.float32(2.1774302e+21), np.float32(0.13777709))

(np.float32(2.1774302e+21), np.float32(0.13777709))

(np.float32(2.177354e+21), np.float32(0.13778192))

In [ ]:
Idirt = Idirt[gc.ICARUS_dirtcut(Idirt)].copy()

for c in Idf.columns:
    if "_univ" in c:
        Idirt[c] = 1
        Ioffbeam[c] = 1

if "dirt" not in Idf.columns:
    Idf["dirt"] = False
    Idirt["dirt"] = True
    Ioffbeam["dirt"] = False
    
Idf = pd.concat([Idf[~Idf.dirt], Idirt])#, Ioffbeam])

In [ ]:
for c in Sdf.columns:
    if "_univ" in c:
        Sdirt[c] = 1
        Soffbeam[c] = 1

if "dirt" not in Sdf.columns:
    Sdf["dirt"] = False
    Sdirt["dirt"] = True
    Soffbeam["dirt"] = False

In [ ]:
loaddf.scale_pot(Sdf, Spot, SGOAL_POT)
loaddf.scale_pot(Sdirt, Sdirtpot, SGOAL_POT)
loaddf.scale_pot(Soffbeam, Soffbeampot, SGOAL_POT)

Sdetvars_small, Sdetvar_pots_small = loaddf.match_common_evts(Sdetvarsmatch_small, Sdetvars_small, Sdetvar_pots_small)
for i in range(len(Sdetvars_small)):
    loaddf.scale_pot(Sdetvars_small[i], Sdetvar_pots_small[i], SGOAL_POT)
    
Sdetvars, Sdetvar_pots = loaddf.match_common_evts(Sdetvarsmatch, Sdetvars, Sdetvar_pots)
for i in range(len(Sdetvars)):
    loaddf.scale_pot(Sdetvars[i], Sdetvar_pots[i], SGOAL_POT)
Sdf = pd.concat([Sdf[~Sdf.dirt], Sdirt, Soffbeam])

In [ ]:
Ichi2_detvars = [syst.v_chi2smear(Idf), syst.v_chi2hi(Idf), syst.v_chi2alpha(Idf), syst.v_chi2beta(Idf), syst.v_chi2R(Idf)]
ICHI2_DETVAR_NAMES = ["Smeared dE/dx",
    "Gain Hi",
    "EMB $\\alpha + 0.008$",
    "EMB $\\beta + 0.008$",
    "EMB $R + 0.02$",
]

In [ ]:
Schi2_detvars = [syst.v_chi2smear(Sdf), syst.v_chi2hi(Sdf),  syst.v_chi2alpha(Sdf),  syst.v_chi2beta(Sdf),  syst.v_chi2R(Sdf)]
SCHI2_DETVAR_NAMES = [
    "Smeared dE/dx",
    "Gain Hi",
    "EMB $\\alpha + 0.008$",
    "EMB $\\beta + 0.008$",
    "EMB $R + 0.02$",
]

In [ ]:
S_TRIGGER_NAMES = ["Trigger"]
I_TRIGGER_NAMES = ["Trigger Run 2", "Trigger Run 4"]

In [ ]:
Idf['selected'] = gc.all_cuts(Idf)
Idf['selected_noflash'] = gc.presel_cut(Idf) & gc.cosmic_cut(Idf) & gc.twoprong_cut(Idf) & gc.pid_cut(Idf)
Ioffbeam['selected'] = gc.all_cuts(Ioffbeam)

for i in range(len(Idetvars)):
    Idetvars[i]['selected'] = gc.all_cuts(Idetvars[i])

for i in range(len(Ichi2_detvars)):
    Ichi2_detvars[i]['selected'] = gc.all_cuts(Ichi2_detvars[i])

In [ ]:
Sdf['selected'] = gc.all_cuts(Sdf)
Sdf['selected_noflash'] = gc.presel_cut(Sdf) & gc.cosmic_cut(Sdf) & gc.twoprong_cut(Sdf) & gc.pid_cut(Sdf)

Soffbeam['selected'] = gc.all_cuts(Soffbeam)

for i in range(len(Sdetvars)):
    Sdetvars[i]['selected'] = gc.all_cuts(Sdetvars[i])

for i in range(len(Sdetvars_small)):
    Sdetvars_small[i]['selected'] = gc.all_cuts(Sdetvars_small[i])
    
for i in range(len(Schi2_detvars)):
    Schi2_detvars[i]['selected'] = gc.all_cuts(Schi2_detvars[i])

In [ ]:
# ============================================================
# TRIGGER SYSTEMATIC: flash-PE scale-factor uncertainty
# ============================================================
# Best-fit MC PE scale factors from the data/MC fits in FlashMCDataComparison.ipynb
# (applied to flash_maxpe at load time in loaddf.py):
#   SBND: 0.642 +/- 0.005   ICARUS Run2: 0.632 +/- 0.024   ICARUS Run4: 0.358 +/- 0.017
#
# Varying the scale s -> s*(1 -+ f), f = unc/s, is equivalent to varying the
# threshold on flash_maxpe: T -> T/(1 -+ f). The flash cut is applied in the
# "selected" definition (NOT the load-time preselection), so both the
# tightened- and loosened-threshold universes are available.
TRIG_PE_SCALE     = {1: 0.642, 2: 0.632, 4: 0.358}  # Run -> best-fit s (matches loaddf.py)
TRIG_PE_SCALE_UNC = {1: 0.005, 2: 0.024, 4: 0.017}  # Run -> unc. on s
TRIG_THRESHOLD    = {1: -1, 2: 5000., 4: 1000.}  # Run -> gump_cuts.flash_cut threshold

def trig_flash_cut_var(df, updn, runs=None):
    """Flash cut with the threshold varied by the +/-1 sigma PE-scale uncertainty.

    updn = +1: scale DOWN 1 sigma -> threshold UP   (fewer events)
    updn = -1: scale UP   1 sigma -> threshold DOWN (more events)
    runs: restrict the variation to these runs (others stay at the CV threshold);
          used to make the ICARUS Run2 / Run4 variations uncorrelated.
    """
    sel = pd.Series(False, index=df.index)
    for run, T in TRIG_THRESHOLD.items():
        f = TRIG_PE_SCALE_UNC[run] / TRIG_PE_SCALE[run]
        Tvar = T / (1 - updn*f) if (runs is None or run in runs) else T
        sel = sel | ((df.Run == run) & (df.flash_maxpe > Tvar))
    return sel

# SBND (all rows Run 1). NB: Sdf/Idf include the (MC) dirt rows, which correctly
# participate in the variation; the (data) offbeam samples correctly do not.
Sdf["selected_trig_up"] = Sdf["selected_noflash"] & trig_flash_cut_var(Sdf, +1)
Sdf["selected_trig_dn"] = Sdf["selected_noflash"] & trig_flash_cut_var(Sdf, -1)

# ICARUS: vary one run at a time -> Run2 and Run4 uncorrelated
Idf["selected_trig_r2_up"] = Idf["selected_noflash"] & trig_flash_cut_var(Idf, +1, runs=[2])
Idf["selected_trig_r2_dn"] = Idf["selected_noflash"] & trig_flash_cut_var(Idf, -1, runs=[2])
Idf["selected_trig_r4_up"] = Idf["selected_noflash"] & trig_flash_cut_var(Idf, +1, runs=[4])
Idf["selected_trig_r4_dn"] = Idf["selected_noflash"] & trig_flash_cut_var(Idf, -1, runs=[4])

In [ ]:
# --- Trigger-variation diagnostics ---
trig_cols = {(1, "SBND"): ("selected_trig_up", "selected_trig_dn"),
             (2, "ICARUS"): ("selected_trig_r2_up", "selected_trig_r2_dn"),
             (4, "ICARUS"): ("selected_trig_r4_up", "selected_trig_r4_dn")}

for det, df in [("SBND", Sdf), ("ICARUS", Idf)]:
    for run in sorted(df.Run.unique()):
        upcol, dncol = trig_cols[(run, det)]
        inrun = (df.Run == run)
        T = TRIG_THRESHOLD[run]
        f = TRIG_PE_SCALE_UNC[run] / TRIG_PE_SCALE[run]
        wcv = df.loc[df.selected & inrun, "glob_scale"].sum()
        wup = df.loc[df[upcol] & inrun, "glob_scale"].sum()
        wdn = df.loc[df[dncol] & inrun, "glob_scale"].sum()
        print("%s Run %i: T = %.0f -> %.1f (up) / %.1f (dn), f = %.2f%%" %
              (det, run, T, T/(1-f), T/(1+f), 100*f))
        print("   selected: %.2f -> %.2f (%+.2f%%) up / %.2f (%+.2f%%) dn" %
              (wcv, wup, 100*(wup/wcv - 1), wdn, 100*(wdn/wcv - 1)))

# Consistency: tightening can only remove events, loosening can only add them
assert not (Sdf.selected_trig_up & ~Sdf.selected).any()
assert not (Sdf.selected & ~Sdf.selected_trig_dn).any()
assert not ((Idf.selected_trig_r2_up | Idf.selected_trig_r4_up) & ~Idf.selected).any()
assert not (Idf.selected & ~(Idf.selected_trig_r2_dn & Idf.selected_trig_r4_dn)).any()

In [ ]:
CONFERENCE = False
dbd = False

if dbd:
    PLOTDIR = "/exp/sbnd/app/users/nrowe/cafpyana/analysis_village/gump/nb/Neutrino26Plots/"
elif CONFERENCE:
    PLOTDIR = "/exp/sbnd/app/users/nrowe/cafpyana/analysis_village/gump/nb/Neutrino26PlotsWhite/"
elif not CONFERENCE:
    PLOTDIR = "/exp/sbnd/app/users/nrowe/cafpyana/analysis_village/gump/nb/WorkshopPlots/"

os.makedirs(PLOTDIR, exist_ok=True)
os.makedirs(PLOTDIR + "/png", exist_ok=True)

In [ ]:
if CONFERENCE:
    p = 0.02
else:
    p = 0.005

Ssystematics = [
    loaddf.FluxSystematic(Sdf),
    loaddf.XSecSystematic(Sdf),
    syst.NormalizationSystematic(p),
    #syst.SystSampleSystematic(Sdf[gc.OOAVSBND(Sdf)]),
    #syst.StatSampleSystematic(Soffbeam, norm=0.1)
    #syst.StatSampleSystematic(Soffbeam, norm=0.1) # TODO: change after unblinding. Simulate scaling up stats by 10x.
]

Isystematics = [
    loaddf.FluxSystematic(Idf),
    loaddf.XSecSystematic(Idf),
    syst.NormalizationSystematic(p),
    #syst.SystSampleSystematic(Idf[gc.OOAVICARUS(Idf)]),
    #syst.StatSampleSystematic(Ioffbeam, norm=0.1) 
    #syst.StatSampleSystematic(Ioffbeam, norm=0.1) # TODO: change after unblinding. Simulate scaling up stats by 10x.
]


systematics = [
    syst.CorrelatedSystematic(loaddf.FluxSystematic(Sdf), loaddf.FluxSystematic(Idf)),
    syst.CorrelatedSystematic(loaddf.XSecSystematic(Sdf), loaddf.XSecSystematic(Idf)),
    # POT norm is correlated SBND Run1 (1e20) - ICARUS Run4 (3e20), and uncorrelated ICARUS Run 2 (2e20)
    syst.SystematicList(
        [
            syst.UnCorrelatedSystematic(syst.NormalizationSystematic(0), syst.NormalizationSystematic(p*(2./5.))),
            syst.CorrelatedSystematic(syst.NormalizationSystematic(p), syst.NormalizationSystematic(p*(3./5.))),
        ]),
    #syst.UnCorrelatedSystematic(syst.StatSampleSystematic(Soffbeam, norm=0.1), syst.StatSampleSystematic(Ioffbeam, norm=0.1)),
]

if CONFERENCE==False:
    Isyst_det = syst.SystematicList([syst.SampleSystematic(d, cvdf=Idetvars[0]) for d in Idetvars[1:]]+\
                                    [syst.SampleSystematic(d) for d in Ichi2_detvars]+\
                                    [
                                        syst.SelectionSystematic(Idf, ["selected_trig_r2_up", "selected_trig_r2_dn"]),
                                        syst.SelectionSystematic(Idf, ["selected_trig_r4_up", "selected_trig_r4_dn"])
                                    ] +\
                                    [syst.SystSampleSystematic(Idf[gc.OOAVICARUS(Idf)])]
                                   )
    
    Ssyst_det = syst.SystematicList([syst.SampleSystematic(d, cvdf=Sdetvars[0]) for d in Sdetvars[1:]]+\
                                    [syst.SampleSystematic(d, cvdf=Sdetvars_small[0]) for d in Sdetvars_small[1:]]+\
                                    [syst.SampleSystematic(d) for d in Schi2_detvars]+\
                                    [syst.SelectionSystematic(Sdf, ["selected_trig_up", "selected_trig_dn"])] +\
                                    [syst.SystSampleSystematic(Sdf[gc.OOAVSBND(Sdf)])]
                                   )
    
    Ssystematics.append(Ssyst_det)
    Isystematics.append(Isyst_det)
    systematics.append(syst.UnCorrelatedSystematic(Ssyst_det, Isyst_det))
elif CONFERENCE==True:
    Isyst_det = syst.SystematicList([syst.SystSampleSystematic(Idf[gc.OOAVICARUS(Idf)])]
                                   )
    
    Ssyst_det = syst.SystematicList([syst.SystSampleSystematic(Sdf[gc.OOAVSBND(Sdf)])]
                                   )
    
    Ssystematics.append(Ssyst_det)
    Isystematics.append(Isyst_det)
    systematics.append(syst.UnCorrelatedSystematic(Ssyst_det, Isyst_det))

labels = [
    "Flux",
    "XSec",
    "POT Norm.",
    # "Offbeam"
]

if CONFERENCE == False:
    labels.append("Detector/Uncorr.")
elif CONFERENCE == True:
    labels.append("Dirt")
labels.extend(["Stat","MC Stat Uncert.","All"]) # these get added 2 cells below

In [ ]:
var = "nu_E_calo"
wgt = "glob_scale"
cut = 'selected'

bins = np.array([0.3, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0, 1.25, 1.5])
centers = (bins[1:] + bins[:-1]) / 2

In [ ]:
SCV = np.histogram(Sdf.loc[Sdf[cut], var], bins=bins, weights=Sdf.loc[Sdf[cut], wgt])[0]
Scovs = [s.cov(var, cut, bins, SCV) for s in Ssystematics]
Scovs.append(np.diag(SCV))

wSCVsq = np.histogram(Sdf.loc[Sdf[cut], var], bins=bins, weights=np.pow(Sdf.loc[Sdf[cut], wgt], 2))[0]
Scovs.append(np.diag(wSCVsq))
Scovs.append(np.sum(Scovs, axis=0))

In [ ]:
ICV = np.histogram(Idf.loc[Idf[cut], var], bins=bins, weights=Idf.loc[Idf[cut], wgt])[0]
Icovs = [s.cov(var, cut, bins, ICV) for s in Isystematics]
Icovs.append(np.diag(ICV))

wICVsq = np.histogram(Idf.loc[Idf[cut], var], bins=bins, weights=np.pow(Idf.loc[Idf[cut], wgt], 2))[0]
Icovs.append(np.diag(wICVsq))
Icovs.append(np.sum(Icovs, axis=0))

In [ ]:
import matplotlib.ticker as ticker
def add_style(ax, xlabel, title="", det="ICARUS", ylabel='Events / $10^{20}$ POT', 
              legend_loc=None, legend_ncol=1, legend_title=None, 
              dabadee=False, transparent=False):
    # Retrieve the figure object from the axes
    fig = ax.get_figure()
    
    # Default color logic
    tick_color = "black"
    font_color = "black"
    bg_color = "white"
    bg_alpha = 1.0

    if dabadee:
        bg_color = 'none'#'#525b83'
        font_color = 'white'
        tick_color = 'white'
    elif transparent:
        bg_color = 'none' # Or (0,0,0,0)
        bg_alpha = 0.0
        font_color = 'white' # Defaulting to white text for transparency
        tick_color = 'white'
    
    # Apply background settings
    fig.patch.set_facecolor(bg_color)
    fig.patch.set_alpha(bg_alpha)
    ax.set_facecolor(bg_color)
    ax.patch.set_alpha(bg_alpha)
    for spine in ax.spines.values():
        spine.set_edgecolor(tick_color)
    ax.tick_params(axis='both', which='major', colors=tick_color)

    
    # Labels and Title
    ax.set_xlabel(xlabel, fontsize=FONTSIZE, fontweight='bold', color=font_color)
    ax.set_ylabel(ylabel, fontsize=FONTSIZE, fontweight='bold', color=font_color)
    ax.set_title(fr"$\bf{{{det}}}$ $\bf{{{title}}}$", fontsize=FONTSIZE+2, color=font_color)

    # Legend
    leg = ax.legend(fontsize=FONTSIZE-1, loc=legend_loc, ncol=legend_ncol, 
                    title=legend_title, title_fontsize=FONTSIZE, labelcolor=font_color)
    
    if transparent and leg:
        leg.get_frame().set_alpha(0.0) # Ensure legend background is also transparent
    elif leg:
        leg.get_frame().set_color(bg_color) # Ensure legend background is also transparent   

In [ ]:
if CONFERENCE == False:
    plt.clf()
    S_detsyst_covs = [s.cov(var, cut, bins, SCV) for s in Ssystematics[labels.index("Detector/Uncorr.")].systs]
    I_detsyst_covs = [s.cov(var, cut, bins, ICV) for s in Isystematics[labels.index("Detector/Uncorr.")].systs]
    
    combined = np.zeros(Scovs[0].shape)
    for c, l in zip(S_detsyst_covs, SDETVAR_NAMES[1:] + SDETVAR_NAMES_SMALL[1:] + SCHI2_DETVAR_NAMES + S_TRIGGER_NAMES):
        _ = plt.hist(centers, bins=bins, weights=(np.sqrt(np.diag(c))/SCV), label=l, histtype="step", linewidth=2)
        combined += c
        
    for c, l in zip(Scovs[4:], labels[4:len(labels)-1]):
        _ = plt.hist(centers, bins=bins, weights=(np.sqrt(np.diag(c))/SCV), label=l, histtype="step", linewidth=2)
        combined += c
    
    plt.hist(centers, bins=bins, weights=(np.sqrt(np.diag(combined))/SCV), label="Total", histtype="step", color="black", linewidth=2, linestyle="--")
    
    add_style(plt.gca(), "Reco. Neutrino Energy [GeV]", det="SBND", ylabel="Fractional Uncertainty", 
              legend_loc="upper center", legend_ncol=2, legend_title="Uncorrelated Uncertainties")
    
    plt.ylim([0, 0.125])
    plt.savefig(PLOTDIR + "png/SBND_signalbox_systematics_uncorr.png", bbox_inches="tight")
    plt.show()
    
    plt.clf()
    combined = np.zeros(Icovs[0].shape)
    for c, l in zip(I_detsyst_covs, IDETVAR_NAMES[1:] + ICHI2_DETVAR_NAMES + I_TRIGGER_NAMES):
        _ = plt.hist(centers, bins=bins, weights=(np.sqrt(np.diag(c))/ICV), label=l, histtype="step", linewidth=2)
        combined += c
        
    for c, l in zip(Icovs[4:], labels[4:len(labels)-1]):
        _ = plt.hist(centers, bins=bins, weights=(np.sqrt(np.diag(c))/ICV), label=l, histtype="step", linewidth=2)
        combined += c
    
    plt.hist(centers, bins=bins, weights=(np.sqrt(np.diag(combined))/ICV), label="Total", histtype="step", color="black", linewidth=2, linestyle="--")
    
    add_style(plt.gca(), "Reco. Neutrino Energy [GeV]", det="ICARUS", ylabel="Fractional Uncertainty", 
              legend_loc="upper center", legend_ncol=2, legend_title="Uncorrelated Uncertainties")
    
    plt.ylim([0, 0.125])
    plt.savefig(PLOTDIR + "png/ICARUS_signalbox_systematics_uncorr.png", bbox_inches="tight")
    plt.show()

In [ ]:
plt.style.use('/exp/sbnd/app/users/nrowe/cafpyana/analysis_village/gump/dune.mplstyle')
for c, l in zip(Scovs, labels):
    # print(l)
    # if ((l != "Detector/Uncorr.") | (CONFERENCE == False)):
    #print(SCV)
    print(np.sqrt(np.diag(c)))
    _ = plt.hist(centers, bins=bins, weights=(np.sqrt(np.diag(c))/SCV), label=l, histtype="step", linewidth=2)

add_style(plt.gca(), "Reco. Neutrino Energy [GeV]", det="SBND", ylabel="Fractional Uncertainty", 
          legend_loc="upper center", legend_ncol=2, dabadee=dbd)

plt_text = "SBN Analysis in Progress\nSBND Simulation"

if dbd:
    color='white'
else:
    color='black'

if CONFERENCE: 
    plt_text+="\nNo Detector Syst. Included"
plt.text(0.75, 0.25, plt_text, color=color, fontsize=FONTSIZE+2, zorder=10)
plt.ylim([0.0001, 0.45])

# plt.yscale("log")

plt.savefig(PLOTDIR + "png/SBND_signalbox_systematics.png", bbox_inches="tight")

In [ ]:
for c, l in zip(Icovs, labels):
    #if l != "Beam Off" and l != "Dirt" and ((l != "Detector/Uncorr.") | (CONFERENCE == False)):
    _ = plt.hist(centers, bins=bins, weights=(np.sqrt(np.diag(c))/ICV), label=l, histtype="step", linewidth=2)

add_style(plt.gca(), "Reco. Neutrino Energy [GeV]", det="ICARUS", ylabel="Fractional Uncertainty", 
          legend_loc="upper center", legend_ncol=2, dabadee=dbd)

plt_text = "SBN Analysis in Progress\nICARUS Simulation"

if dbd:
    color='white'
else:
    color='black'

if CONFERENCE: plt_text+="\nNo Detector Syst. Included"
plt.text(0.75, 0.25, plt_text, color=color, fontsize=FONTSIZE+2, zorder=10)
plt.ylim([0.0001, 0.45])
# plt.yscale("log")
print(PLOTDIR + "png/ICARUS_signalbox_systematics.png")
plt.savefig(PLOTDIR + "png/ICARUS_signalbox_systematics.png", bbox_inches="tight")

In [ ]:
covs = [s.cov(var, cut, bins, np.concatenate((SCV, ICV))) for s in systematics]
covs.append(np.diag(np.concatenate((SCV, ICV))))
covs.append(np.diag(np.concatenate((wSCVsq, wICVsq))))

In [ ]:
def ratio_cov_full(x, y, cov):
    """
    Covariance of r = x / y given the full covariance of (x, y).

    Parameters
    ----------
    x, y : array-like, shape (n,)
        Central values
    cov : array-like, shape (2n, 2n)
        Full covariance matrix of (x, y)

        Ordering must be:
        cov = [[Cov(x,x), Cov(x,y)],
               [Cov(y,x), Cov(y,y)]]

    Returns
    -------
    cov_r : ndarray, shape (n, n)
        Covariance matrix of r
    """
    n = len(x)
    assert cov.shape == (2*n, 2*n)

    # Protect against division by zero
    eps = 1e-12
    y_safe = np.where(np.abs(y) < eps, eps, y)

    Dx = np.diag(1.0 / y_safe)
    Dy = np.diag(-x / y_safe**2)

    # Full Jacobian: shape (n, 2n)
    J = np.hstack([Dx, Dy])

    return J @ cov @ J.T

In [ ]:
ratio = SCV / ICV
ratio_cov_flux = ratio_cov_full(SCV, ICV, covs[0])
ratio_cov_xsec = ratio_cov_full(SCV, ICV, covs[1])
ratio_cov_norm = ratio_cov_full(SCV, ICV, covs[2])
ratio_cov_det = ratio_cov_full(SCV, ICV, covs[3])
ratio_cov_stat = ratio_cov_full(SCV, ICV, covs[4])
ratio_cov_mcstat = ratio_cov_full(SCV, ICV, covs[5])

ratio_cov_all = ratio_cov_full(SCV, ICV, np.sum(covs, axis=0))

In [ ]:
_ = plt.hist(centers, bins=bins, weights=np.sqrt(np.diag(ratio_cov_flux))/ratio, histtype="step", linewidth=2, label="Flux")
_ = plt.hist(centers, bins=bins, weights=np.sqrt(np.diag(ratio_cov_xsec))/ratio, histtype="step", linewidth=2, label="XSec")
_ = plt.hist(centers, bins=bins, weights=np.sqrt(np.diag(ratio_cov_norm))/ratio, histtype="step", linewidth=2, label="POT Norm.")
if (CONFERENCE == False):
    _ = plt.hist(centers, bins=bins, weights=np.sqrt(np.diag(ratio_cov_det))/ratio, histtype="step", linewidth=2, label="Detector/Uncorr.")
elif (CONFERENCE == True):
    _ = plt.hist(centers, bins=bins, weights=np.sqrt(np.diag(ratio_cov_det))/ratio, histtype="step", linewidth=2, label="Dirt")
_ = plt.hist(centers, bins=bins, weights=np.sqrt(np.diag(ratio_cov_stat))/ratio, histtype="step", linewidth=2, label="Stat.")
_ = plt.hist(centers, bins=bins, weights=np.sqrt(np.diag(ratio_cov_mcstat))/ratio, histtype="step", linewidth=2, label="MC Stat Uncert.")
_ = plt.hist(centers, bins=bins, weights=np.sqrt(np.diag(ratio_cov_all))/ratio, histtype="step", linewidth=2, label="All")
# plt.ylim([0, 0.25])

if dbd:
    color='white'
else:
    color='black'
    
add_style(plt.gca(), "Reco. Neutrino Energy [GeV]", title="Ratio", det="SBND / ICARUS", 
          ylabel="Fractional Uncertainty \n on SBND/ICARUS Ratio", legend_loc="upper center", legend_ncol=2, dabadee=dbd)

plt_text = "SBN Analysis in Progress\nSBN Simulation"
if CONFERENCE: plt_text+="\nNo Detector Syst. Included"
plt.text(0.7, 0.1, plt_text, color=color, fontsize=FONTSIZE+2)
plt.ylim([0.0001, 0.23])

plt.savefig(PLOTDIR + "png/ratio_signalbox_systematics.png", bbox_inches="tight")